# receipt-vlm on a rented GPU (Colab / Kaggle / cloud)

All logic lives in `src/receipt_vlm/`; this notebook only runs the same `make` targets as on any
machine. Target: one 16 GB GPU (T4/L4) with `configs/*/gpu16.yaml`, or ≥ 24 GB with
`gpu24plus`. Order matters: **validation for every decision, test once at the end**.

Secrets: set `HF_TOKEN` (needed only to push models) and optionally `WANDB_API_KEY` in the Colab
secrets panel.

On Colab, predictions and metrics are written straight to Google Drive
(`MyDrive/receipt-vlm-results`) and the trained adapters are copied there after sections 3
and 4. After a disconnect, re-run the notebook from the top and skip the finished sections:
an interrupted evaluation resumes where it stopped.

In [ ]:
# Colab: keep results on Google Drive (a free session can disconnect and wipe /content)
OUT = "/content"
try:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive"
except ImportError:
    pass

In [ ]:
!nvidia-smi
import os

REPO = "https://github.com/StepanovArt/expense-tracker"
BRANCH = "feature/receipt-vlm"  # or main once merged
!git clone -b {BRANCH} {REPO} /content/expense-tracker
%cd /content/expense-tracker/receipt-vlm

In [ ]:
# secrets (optional)
import contextlib

try:
    from google.colab import userdata

    for k in ("HF_TOKEN", "WANDB_API_KEY"):
        with contextlib.suppress(Exception):
            os.environ[k] = userdata.get(k)
except ImportError:
    pass
!pip -q install uv
!make setup-gpu
# Tesseract binary + Arabic/Russian traineddata for baseline B0
!apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-ara tesseract-ocr-rus

In [ ]:
# Keep results on Drive: predictions are appended line by line, so an evaluation cut off by
# a disconnect resumes where it stopped when its cell is run again on a new machine.
import shutil
import subprocess
from pathlib import Path

SAVE = f"{OUT}/receipt-vlm-results"
for d in ("preds", "metrics"):
    local, remote = Path("outputs") / d, Path(SAVE) / d
    remote.mkdir(parents=True, exist_ok=True)
    if not local.is_symlink():
        if local.is_dir():
            shutil.copytree(local, remote, dirs_exist_ok=True)
            shutil.rmtree(local)
        local.parent.mkdir(parents=True, exist_ok=True)
        local.symlink_to(remote)
# adapters trained in an earlier session: if listed below, skip sections 3 / 4
for d in ("sft/best", "grpo/best"):
    if (Path(SAVE) / d).is_dir() and not (Path("outputs") / d).is_dir():
        shutil.copytree(Path(SAVE) / d, Path("outputs") / d)
if Path("outputs/grpo/best").is_dir() and not Path("outputs/sft/merged").is_dir():
    subprocess.run(
        "uv run python scripts/merge_lora.py --adapter outputs/sft/best --out outputs/sft/merged",
        shell=True,
        check=True,
    )
print("predictions on Drive:", sorted(p.name for p in (Path(SAVE) / "preds").iterdir()))
print("adapters:", [d for d in ("sft/best", "grpo/best") if (Path("outputs") / d).is_dir()])

## 1. Data (Phase 1–2)
Adds CORD v2 (HF Hub) and WildReceipt to the frozen SROIE + synthetic splits; existing test entries cannot move (the build fails if they would).

In [ ]:
%cd /content/expense-tracker/receipt-vlm
!make data
!make synth
!uv run pytest -m data -q
!cat reports/data_stats.md

## 2. Zero-shot baselines on validation (Phase 3)

In [ ]:
%cd /content/expense-tracker/receipt-vlm
!make eval-baselines SPLIT=val
!make report

## 3. SFT (Phase 4)
Checkpoint selection by generation-based val F1 on a fixed 200-image subset. Try `lora.train_projector=true` as a second run and keep whichever wins on val.

In [ ]:
%cd /content/expense-tracker/receipt-vlm
!make sft CONFIG=configs/sft/gpu16.yaml
!mkdir -p {SAVE}/sft && cp -r outputs/sft/best {SAVE}/sft/  # save the adapter before evaluating
# optional comparison run:
# !uv run python scripts/train_sft.py --config configs/sft/gpu16.yaml output_dir=outputs/sft_proj lora.train_projector=true
!make eval MODEL=M1 SPLIT=val EVALCFG=configs/eval/final.yaml
!make report

## 4. GRPO (Phase 6) — validation first

In [ ]:
%cd /content/expense-tracker/receipt-vlm
!make grpo CONFIG=configs/grpo/gpu16.yaml
!mkdir -p {SAVE}/grpo && cp -r outputs/grpo/best {SAVE}/grpo/  # save the adapter before evaluating
!make eval MODEL=M2 SPLIT=val EVALCFG=configs/eval/final.yaml
!make report

## 5. Frozen test set — run once (Phase 5/6)

In [ ]:
%cd /content/expense-tracker/receipt-vlm
!uv run python scripts/run_eval.py --config configs/eval/final.yaml --split test --models B0 B1 B2 B3 M1 M2
!make report
!uv run python scripts/error_analysis.py --model M2 --split test

## 6. Save results back to the repo
Predictions stay in `outputs/` (gitignored, large); metrics, report and figures are committed so every number is reproducible from a command.

In [ ]:
%cd /content/expense-tracker/receipt-vlm
!git add reports data/splits && git -c user.name=colab -c user.email=colab@localhost commit -m "results: GPU run" || true
# !git push   # needs a token with push access
!tar czhf {OUT}/receipt-vlm-outputs.tgz outputs/preds outputs/metrics outputs/sft/best outputs/grpo/best

## 7. Release (Phase 8)

In [ ]:
%cd /content/expense-tracker/receipt-vlm
!make merge
!uv run python scripts/push_to_hub.py --adapter outputs/sft/best --merged outputs/merged --repo <your-hf-user>/qwen3-vl-2b-receipts